# LeetCode 714 — Best Time to Buy and Sell Stock with Transaction Fee bằng Python

Link đề: https://leetcode.com/problems/best-time-to-buy-and-sell-stock-with-transaction-fee/ · Độ khó: **Medium** · Chủ đề: Greedy, Array, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def maxProfit(self, prices: List[int], fee: int) -> int:
        ...
```

Ghi chú: các ví dụ ngoài hai ví dụ của đề (ví dụ 3 và 4 ở mục 2) do tôi tự đặt, đáp án đã kiểm chứng bằng vét cạn; nếu cần khớp từng chữ, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có giá cổ phiếu từng ngày `prices`, được **mua/bán nhiều lần** nhưng mỗi giao dịch (một lần mua + một lần bán) mất **phí `fee`**; không được cầm hai cổ phiếu cùng lúc; tìm **lợi nhuận lớn nhất**.
2. Nắm các công cụ Python: gán đồng thời `a, b = x, y`, `max` hai/ba đối số, `float("-inf")`, `functools.lru_cache` (nhớ kết quả đệ quy), danh sách 2 chiều/mảng song song, và quy ước "tách việc cập nhật trạng thái thành hàm nhỏ".
3. Đi từ **vét cạn mọi cách mua/bán (đệ quy)** → **đệ quy có nhớ (memo) theo trạng thái (ngày, đang cầm hay không)** → **bảng DP hai trạng thái `cash`/`hold`** → **rút gọn còn hai biến O(1)** → **tham lam với "giá mua hiệu dụng"**.
4. Hiểu mấu chốt: tại mỗi ngày chỉ có **hai trạng thái** — *không cầm cổ phiếu* (`cash` = tiền tối đa) và *đang cầm* (`hold` = tiền tối đa khi đang cầm một cổ phiếu) — và mỗi ngày ta chỉ chọn giữa vài chuyển trạng thái; đây là **máy trạng thái**.
5. Tránh các lỗi kinh điển: trừ phí hai lần, khởi tạo `hold = 0` (cổ phiếu miễn phí), trả `hold` thay vì `cash`, dùng lời giải bài "không phí" (cộng mọi mức tăng) mà quên phí, và tham lam quên "đặt lại giá mua" sau khi bán.

## 2. Đề bài & ví dụ

Cho mảng `prices` với `prices[i]` là giá cổ phiếu ngày `i`, và số nguyên `fee` là **phí giao dịch**.

Hãy tìm **lợi nhuận tối đa**. Bạn có thể thực hiện **nhiều giao dịch tùy ý**, nhưng mỗi giao dịch phải trả phí `fee`.

Lưu ý:

- Không được tham gia **nhiều giao dịch cùng lúc**: phải bán cổ phiếu trước khi mua lại (có thể bán rồi mua lại **ngay trong cùng một ngày**).
- Phí chỉ tính **một lần cho mỗi cặp mua – bán**.

**Ví dụ 1** (của đề)

- Input: `prices = [1,3,2,8,4,9]`, `fee = 2`
- Output: `8`
- Giải thích: mua ở `prices[0] = 1`, bán ở `prices[3] = 8`; mua ở `prices[4] = 4`, bán ở `prices[5] = 9`. Tổng lãi `((8 - 1) - 2) + ((9 - 4) - 2) = 8`.

**Ví dụ 2** (của đề)

- Input: `prices = [1,3,7,5,10,3]`, `fee = 3`
- Output: `6`

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `prices = [5]`, `fee = 1`
- Output: `0`
- Giải thích: chỉ có một ngày nên không thể mua rồi bán.

**Ví dụ 4 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `prices = [1,5,10]`, `fee = 1`
- Output: `8`
- Giải thích: mua ngày 0 (giá `1`), bán ngày 2 (giá `10`): lãi `10 - 1 - 1 = 8`. Nếu chia làm hai giao dịch `1 -> 5` và `5 -> 10` thì lãi `3 + 4 = 7` (trả phí hai lần), kém hơn.

**Ràng buộc:**

- `1 <= prices.length <= 5 * 10^4`
- `1 <= prices[i] < 5 * 10^4`
- `0 <= fee < 5 * 10^4`

In [1]:
from typing import List
from functools import lru_cache
import random
import sys
import time

examples = [
    ([1, 3, 2, 8, 4, 9], 2, 8),
    ([1, 3, 7, 5, 10, 3], 3, 6),
    ([5], 1, 0),
    ([1, 5, 10], 1, 8),
]
for prices, fee, expected in examples:
    print(prices, "fee =", fee, "-> kỳ vọng", expected)

[1, 3, 2, 8, 4, 9] fee = 2 -> kỳ vọng 8
[1, 3, 7, 5, 10, 3] fee = 3 -> kỳ vọng 6
[5] fee = 1 -> kỳ vọng 0
[1, 5, 10] fee = 1 -> kỳ vọng 8


## 3. Kiến thức nền

### 3.1. Lợi nhuận một giao dịch và phí

Một giao dịch gồm **mua ở ngày `i`** rồi **bán ở ngày `j > i`**: lãi gộp là `prices[j] - prices[i]`, lãi ròng là `prices[j] - prices[i] - fee`. Tổng lãi của nhiều giao dịch **không chồng nhau** là tổng các lãi ròng. Nếu lãi ròng âm thì ta đơn giản **không làm** giao dịch đó.

In [2]:
prices, fee = [1, 3, 2, 8, 4, 9], 2
trades = [(0, 3), (4, 5)]                         # (ngày mua, ngày bán)
profits = [prices[j] - prices[i] - fee for i, j in trades]
print(profits, "tổng =", sum(profits))            # [5, 3] tổng = 8
print("một giao dịch tệ:", prices[2] - prices[1] - fee, "(âm -> bỏ qua)")

[5, 3] tổng = 8
một giao dịch tệ: -3 (âm -> bỏ qua)


### 3.2. Máy trạng thái: "đang cầm" hay "không cầm"

Cuối mỗi ngày, ta ở một trong **hai trạng thái**:

- **`cash`**: không cầm cổ phiếu; giá trị là **số tiền tối đa** ta có.
- **`hold`**: đang cầm một cổ phiếu; giá trị là **số tiền tối đa** (tính theo "tiền mặt", có thể âm vì đã bỏ tiền mua) ta có.

Mỗi ngày với giá `p`, các **chuyển trạng thái** có thể:

```
cash mới = max( cash,            # tiếp tục không cầm
                hold + p - fee ) # bán cổ phiếu đang cầm, trả phí
hold mới = max( hold,            # tiếp tục cầm
                cash - p )       # mua cổ phiếu bằng tiền đang có
```

Vì mua/bán trong cùng ngày không có lợi (chỉ mất phí) nên dùng `cash` cũ hay mới đều cho kết quả như nhau; ta sẽ kiểm chứng ở mục 12. Đáp án cuối cùng là `cash` ở ngày cuối (cầm cổ phiếu dở dang không tính).

In [3]:
def step(cash, hold, p, fee):
    """Chuyển trạng thái một ngày với giá p; trả về (cash mới, hold mới)."""
    new_cash = max(cash, hold + p - fee)
    new_hold = max(hold, cash - p)
    return new_cash, new_hold


cash, hold = 0, float("-inf")        # trước ngày đầu: chưa có gì, chưa thể đang cầm
for p in [1, 3]:
    cash, hold = step(cash, hold, p, 2)
    print(f"giá {p}: cash = {cash}, hold = {hold}")

giá 1: cash = 0, hold = -1
giá 3: cash = 0, hold = -1


### 3.3. Gán đồng thời (tuple unpacking)

`a, b = x, y` tính **cả vế phải trước** rồi mới gán. Nhờ vậy `cash, hold = max(cash, hold + p - fee), max(hold, cash - p)` dùng **giá trị cũ** của cả hai biến ở vế phải — đúng điều ta muốn khi cập nhật hai trạng thái cùng lúc.

In [4]:
a, b = 1, 2
a, b = b, a                           # hoán đổi không cần biến tạm
print(a, b)                           # 2 1

cash, hold, p, fee = 0, -1, 8, 2
cash, hold = max(cash, hold + p - fee), max(hold, cash - p)   # cả hai dùng giá trị CŨ
print(cash, hold)                     # 5 -1

2 1
5 -1


### 3.4. `float("-inf")`

`float("-inf")` là âm vô cực: nhỏ hơn mọi số. Dùng làm giá trị khởi tạo cho "trạng thái chưa thể xảy ra" (như đang cầm cổ phiếu trước ngày đầu). Phép cộng `-inf + p` vẫn là `-inf`, và `max(-inf, x) = x`.

In [5]:
NEG = float("-inf")
print(NEG + 5, max(NEG, -7), NEG < -10**9)

-inf -7 True


### 3.5. Đệ quy có nhớ (`lru_cache`)

Hàm đệ quy `f(i, holding)` gọi lặp lại nhiều lần với cùng đối số. Gắn `@lru_cache(None)` (hoặc `functools.cache`) để Python **nhớ** kết quả của mỗi bộ đối số: lần gọi sau trả ngay. Biến thời gian từ cấp số nhân thành tuyến tính theo số trạng thái.

In [6]:
calls = 0

@lru_cache(maxsize=None)
def fib(n):
    global calls
    calls += 1
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(30), "số lần thân hàm thực sự chạy:", calls)       # 31, thay vì ~2.7 triệu

832040 số lần thân hàm thực sự chạy: 31


### 3.6. Giới hạn đệ quy

Python chỉ cho đệ quy sâu khoảng `1000` tầng mặc định. Bài có `n` tới `5 × 10^4` nên đệ quy theo ngày (`i` từ `0` đến `n`) có thể gây `RecursionError`; ta sẽ thấy cách tránh bằng DP lặp ở các cách sau.

### 3.7. "Tham lam" và "giá mua hiệu dụng"

Ý tưởng tham lam của bài: coi **chi phí mua hiệu dụng** = `giá mua + phí`. Khi gặp giá `p` cao hơn chi phí đó, ta bán và "thu lãi tạm", nhưng **chưa chắc nên bán dứt khoát** — nếu sau này giá còn tăng ta có thể "hủy" lần bán bằng cách coi như vẫn đang cầm với chi phí mới là `p` (mua lại tại `p` không mất phí vì phí đã nằm trong lần bán trước). Phần này được dạy kỹ ở cách 5.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét `prices = [1, 3, 2, 8, 4, 9]`, `fee = 2`. Lần theo máy trạng thái `(cash, hold)`:

- Trước ngày đầu: `cash = 0`, `hold = -inf`.
- Ngày 0, giá `1`: `cash = max(0, -inf + 1 - 2) = 0`; `hold = max(-inf, 0 - 1) = -1`.
- Ngày 1, giá `3`: `cash = max(0, -1 + 3 - 2) = 0`; `hold = max(-1, 0 - 3) = -1`.
- Ngày 2, giá `2`: `cash = max(0, -1 + 2 - 2) = 0`; `hold = max(-1, 0 - 2) = -1`.
- Ngày 3, giá `8`: `cash = max(0, -1 + 8 - 2) = 5`; `hold = max(-1, 0 - 8) = -1`.
- Ngày 4, giá `4`: `cash = max(5, -1 + 4 - 2) = 5`; `hold = max(-1, 5 - 4) = 1` (bán ở `8` rồi mua lại ở `4`).
- Ngày 5, giá `9`: `cash = max(5, 1 + 9 - 2) = 8`; `hold = max(1, 5 - 9) = 1`.

Đáp án `cash = 8`. Ta thấy không cần nhớ "đã mua ngày nào" — chỉ cần **hai con số**. Cell dưới chạy lại đúng các phép tính này.

In [7]:
prices, fee = [1, 3, 2, 8, 4, 9], 2
cash, hold = 0, float("-inf")
print(f"{'ngày':>4} {'giá':>4} {'cash':>5} {'hold':>5}")
for day, p in enumerate(prices):
    cash, hold = step(cash, hold, p, fee)
    print(f"{day:>4} {p:>4} {cash:>5} {hold:>5}")
print("đáp án:", cash)

ngày  giá  cash  hold
   0    1     0    -1
   1    3     0    -1
   2    2     0    -1
   3    8     5    -1
   4    4     5     1
   5    9     8     1
đáp án: 8


## 5. Cách 1 — Brute Force / baseline: đệ quy thử mọi cách mua/bán

**Ý tưởng.** Hàm `go(i, holding)` = lợi nhuận tối đa có thể kiếm được **từ ngày `i` trở đi**, biết hiện đang cầm cổ phiếu (`holding = True`) hay không. Tại ngày `i` ta có hai lựa chọn:

- **Không làm gì**: `go(i + 1, holding)`.
- Nếu **không cầm**: có thể **mua** → trả `prices[i]`: `-prices[i] + go(i + 1, True)`.
- Nếu **đang cầm**: có thể **bán** → nhận `prices[i]` và trả phí: `prices[i] - fee + go(i + 1, False)`.

(Ta tính phí lúc **bán**; tính lúc mua cho cùng đáp án vì mọi lần mua có lợi đều đi kèm một lần bán.) Khi `i == n`: trả `0` (không còn ngày nào; nếu còn đang cầm thì coi như không bán — sẽ không có lợi).

In [8]:
def profit_brute(prices, fee):
    n = len(prices)

    def go(i, holding):
        if i == n:
            return 0
        best = go(i + 1, holding)                                  # không làm gì
        if holding:
            best = max(best, prices[i] - fee + go(i + 1, False))   # bán
        else:
            best = max(best, -prices[i] + go(i + 1, True))         # mua
        return best

    return go(0, False)


for prices, fee, expected in examples:
    got = profit_brute(prices, fee)
    print(prices, fee, "->", got)
    assert got == expected

[1, 3, 2, 8, 4, 9] 2 -> 8
[1, 3, 7, 5, 10, 3] 3 -> 6
[5] 1 -> 0
[1, 5, 10] 1 -> 8


**Độ phức tạp.** Mỗi ngày có tối đa hai nhánh nên thời gian **O(2^n)**; bộ nhớ **O(n)** cho ngăn xếp đệ quy. Với `n = 5 × 10^4` hoàn toàn không dùng được (chỉ dùng cho `n` cỡ 20).

**Nút thắt.** Rất nhiều nhánh khác nhau dẫn tới **cùng trạng thái** `(i, holding)`: dù đường đi trước đó thế nào, tương lai chỉ phụ thuộc vào ngày hiện tại và việc có đang cầm hay không. Chỉ có `2n` trạng thái khác nhau — hãy nhớ lại kết quả.

## 6. Ý tưởng thuật toán / bước đột phá

Bước đột phá là nhận ra **trạng thái nhỏ gọn**:

> Tương lai chỉ phụ thuộc vào **(ngày `i`, đang cầm hay không)**. Số trạng thái là `2n`, mỗi trạng thái có `O(1)` lựa chọn.

Từ đó có chuỗi cải tiến:

1. **Nhớ kết quả đệ quy (memo)**: `O(n)` thời gian, vẫn dùng đệ quy (cách 2).
2. **Lập bảng bottom-up** `cash[i]`, `hold[i]` (cách 3): bỏ đệ quy.
3. **Chỉ cần ngày trước** → hai biến `cash`, `hold` (cách 4): `O(1)` bộ nhớ.
4. **Nhìn bằng "giá mua hiệu dụng"** (cách 5): một biến `buy` và một biến `profit`; rút gọn thành tham lam.

Cả bốn cách đều thực chất là cùng một máy trạng thái.

## 7. Các cách tiếp theo

Tổng cộng 5 cách: cách 1 (vét cạn, mục 5) và bốn cách dưới đây.

### 7.1. Cách 2 — Đệ quy có nhớ (top-down DP)

**Giải thích từng đoạn.** Dùng nguyên hàm `go(i, holding)` của cách 1 và thêm `@lru_cache(None)` (hoặc dict `memo`). Vì `i` và `holding` đều hashable nên cache hoạt động ngay. Số trạng thái `2n`, mỗi trạng thái `O(1)`.

In [9]:
def profit_memo(prices, fee):
    n = len(prices)

    @lru_cache(maxsize=None)
    def go(i, holding):
        if i == n:
            return 0
        best = go(i + 1, holding)
        if holding:
            best = max(best, prices[i] - fee + go(i + 1, False))
        else:
            best = max(best, -prices[i] + go(i + 1, True))
        return best

    return go(0, False)


for prices, fee, expected in examples:
    got = profit_memo(prices, fee)
    print(prices, fee, "->", got)
    assert got == expected

[1, 3, 2, 8, 4, 9] 2 -> 8
[1, 3, 7, 5, 10, 3] 3 -> 6
[5] 1 -> 0
[1, 5, 10] 1 -> 8


**Code nộp LeetCode (cách 2):** (với `n` lên tới `5 × 10^4` cần nâng giới hạn đệ quy; ta để dòng `sys.setrecursionlimit` cho an toàn.)

In [10]:
class Solution2:
    def maxProfit(self, prices: List[int], fee: int) -> int:
        n = len(prices)
        sys.setrecursionlimit(max(sys.getrecursionlimit(), n + 100))

        @lru_cache(maxsize=None)
        def dfs(i: int, holding: int) -> int:
            if i >= n:
                return 0
            ans = dfs(i + 1, holding)
            if holding:
                ans = max(ans, prices[i] - fee + dfs(i + 1, 0))
            else:
                ans = max(ans, -prices[i] + dfs(i + 1, 1))
            return ans

        return dfs(0, 0)


for prices, fee, expected in examples:
    assert Solution2().maxProfit(prices, fee) == expected
print("Solution2 OK")

Solution2 OK


**Độ phức tạp cách 2.** Thời gian **O(n)** (`2n` trạng thái, mỗi trạng thái `O(1)`); bộ nhớ **O(n)** (bảng nhớ + ngăn xếp đệ quy sâu `n`). Rủi ro: đệ quy sâu `5 × 10^4` tầng có thể tràn ngăn xếp C trên một số môi trường dù đã tăng giới hạn đệ quy.

### 7.2. Cách 3 — Bảng DP bottom-up `cash[i]`, `hold[i]`

**Ý tưởng.** Đi **xuôi theo ngày**, lưu hai mảng:

- `cash[i]`: tiền tối đa vào **cuối ngày `i`** khi **không cầm** cổ phiếu.
- `hold[i]`: tiền tối đa vào cuối ngày `i` khi **đang cầm** cổ phiếu.

Khởi tạo ngày 0: `cash[0] = 0`, `hold[0] = -prices[0]` (mua ngay ngày 0). Với `i ≥ 1`:

```
cash[i] = max(cash[i-1], hold[i-1] + prices[i] - fee)
hold[i] = max(hold[i-1], cash[i-1] - prices[i])
```

Đáp án `cash[n-1]`.

In [11]:
def profit_table(prices, fee, verbose=False):
    n = len(prices)
    cash = [0] * n
    hold = [0] * n
    cash[0], hold[0] = 0, -prices[0]
    for i in range(1, n):
        cash[i] = max(cash[i - 1], hold[i - 1] + prices[i] - fee)
        hold[i] = max(hold[i - 1], cash[i - 1] - prices[i])
    if verbose:
        print("cash:", cash)
        print("hold:", hold)
    return cash[-1]


for prices, fee, expected in examples:
    got = profit_table(prices, fee, verbose=(prices == examples[0][0]))
    assert got == expected
print("OK")

cash: [0, 0, 0, 5, 5, 8]
hold: [-1, -1, -1, -1, 1, 1]
OK


**Code nộp LeetCode (cách 3):**

In [12]:
class Solution3:
    def maxProfit(self, prices: List[int], fee: int) -> int:
        n = len(prices)
        cash = [0] * n
        hold = [0] * n
        hold[0] = -prices[0]
        for i in range(1, n):
            cash[i] = max(cash[i - 1], hold[i - 1] + prices[i] - fee)
            hold[i] = max(hold[i - 1], cash[i - 1] - prices[i])
        return cash[-1]


for prices, fee, expected in examples:
    assert Solution3().maxProfit(prices, fee) == expected
print("Solution3 OK")

Solution3 OK


**Độ phức tạp cách 3.** Thời gian **O(n)**, bộ nhớ **O(n)** (hai mảng). Không đệ quy nên chạy được với `n = 5 × 10^4`.

### 7.3. Cách 4 — Rút gọn còn hai biến (O(1) bộ nhớ)

**Ý tưởng.** Ngày `i` chỉ cần `cash[i-1]` và `hold[i-1]`, nên thay hai mảng bằng hai biến, cập nhật **đồng thời** bằng phép gán tuple.

In [13]:
def profit_two_vars(prices, fee, verbose=False):
    cash, hold = 0, -prices[0]
    for p in prices[1:]:
        cash, hold = max(cash, hold + p - fee), max(hold, cash - p)
        if verbose:
            print(f"  giá {p}: cash = {cash}, hold = {hold}")
    return cash


for prices, fee, expected in examples:
    got = profit_two_vars(prices, fee, verbose=(prices == examples[1][0]))
    assert got == expected
print("OK")

  giá 3: cash = 0, hold = -1
  giá 7: cash = 3, hold = -1
  giá 5: cash = 3, hold = -1
  giá 10: cash = 6, hold = -1
  giá 3: cash = 6, hold = 3
OK


**Code nộp LeetCode (cách 4):**

In [14]:
class Solution4:
    def maxProfit(self, prices: List[int], fee: int) -> int:
        cash, hold = 0, -prices[0]
        for p in prices[1:]:
            cash, hold = max(cash, hold + p - fee), max(hold, cash - p)
        return cash


for prices, fee, expected in examples:
    assert Solution4().maxProfit(prices, fee) == expected
print("Solution4 OK")

Solution4 OK


**Độ phức tạp cách 4.** Thời gian **O(n)**, bộ nhớ **O(1)**.

### 7.4. Cách 5 — Tham lam với "giá mua hiệu dụng"

**Ý tưởng.** Giữ biến `buy` = **chi phí hiệu dụng của cổ phiếu ta đang "coi như" cầm** (giá mua + phí) và `profit` = lãi đã chốt. Ban đầu `buy = prices[0] + fee`. Với mỗi giá `p`:

1. **`p + fee < buy`**: có cơ hội mua **rẻ hơn** → cập nhật `buy = p + fee` (bỏ lần mua trước, chưa bán gì).
2. **`p > buy`**: bán có lãi `p - buy` → `profit += p - buy`, rồi đặt **`buy = p`**. Ý nghĩa: ta "bán" nhưng **giữ quyền hủy**: nếu ngày sau giá còn cao hơn `p`, ta chốt thêm phần chênh `p' - p` như thể chưa từng bán (không phải trả phí lần nữa, vì phí đã nằm trong lần bán đầu); còn nếu giá tụt xuống dưới `p - fee` ta có thể mua lại.
3. Ngược lại: không làm gì.

Cách 5 gọn nhưng **khó chứng minh bằng mắt**, nên ta kiểm chứng kỹ bằng vét cạn ở mục 12.

In [15]:
def profit_greedy(prices, fee, verbose=False):
    buy, profit = prices[0] + fee, 0
    for p in prices[1:]:
        if p + fee < buy:
            buy = p + fee
            act = f"mua rẻ hơn -> buy = {buy}"
        elif p > buy:
            gain = p - buy
            profit += gain
            buy = p
            act = f"chốt lãi {gain} -> profit = {profit}, buy = {buy}"
        else:
            act = "không làm gì"
        if verbose:
            print(f"  giá {p}: {act}")
    return profit


for prices, fee, expected in examples:
    got = profit_greedy(prices, fee, verbose=(prices == examples[0][0]))
    assert got == expected
print("OK")

  giá 3: không làm gì
  giá 2: không làm gì
  giá 8: chốt lãi 5 -> profit = 5, buy = 8
  giá 4: mua rẻ hơn -> buy = 6
  giá 9: chốt lãi 3 -> profit = 8, buy = 9
OK


**Code nộp LeetCode (cách 5):**

In [16]:
class Solution5:
    def maxProfit(self, prices: List[int], fee: int) -> int:
        buy, profit = prices[0] + fee, 0
        for p in prices[1:]:
            if p + fee < buy:
                buy = p + fee
            elif p > buy:
                profit += p - buy
                buy = p
        return profit


for prices, fee, expected in examples:
    assert Solution5().maxProfit(prices, fee) == expected
print("Solution5 OK")

Solution5 OK


**Độ phức tạp cách 5.** Thời gian **O(n)**, bộ nhớ **O(1)**. Về bản chất, `buy` tương ứng với `cash - hold` trong DP; đó là lý do tham lam này đúng.

## 8. TRACE từng bước

### 8.1. Trace cách 4 (`cash`, `hold`) trên ví dụ 1

Bảng trạng thái (ví dụ 1: `[1,3,2,8,4,9]`, `fee = 2`):

| Cột | Ý nghĩa |
|---|---|
| `p` | giá hôm nay |
| `bán?` | giá trị `hold + p - fee`: nếu bán hôm nay thì có bao nhiêu tiền |
| `mua?` | giá trị `cash - p`: nếu mua hôm nay thì "tiền đang cầm cổ phiếu" là bao nhiêu |
| `cash`, `hold` | trạng thái sau khi chọn max |

In [17]:
def trace_dp(prices, fee):
    cash, hold = 0, -prices[0]
    print(f"ngày 0 giá {prices[0]}: cash = {cash}, hold = {hold}  (mua ngay: hold = -{prices[0]})")
    for day, p in enumerate(prices[1:], 1):
        sell_val = hold + p - fee
        buy_val = cash - p
        new_cash, new_hold = max(cash, sell_val), max(hold, buy_val)
        why_c = "bán" if sell_val > cash else "giữ"
        why_h = "mua" if buy_val > hold else "giữ"
        print(f"ngày {day} giá {p}: bán? {sell_val:>3} vs cash {cash:>3} -> {new_cash:>3} ({why_c}) | "
              f"mua? {buy_val:>3} vs hold {hold:>3} -> {new_hold:>3} ({why_h})")
        cash, hold = new_cash, new_hold
    return cash

print("kết quả:", trace_dp([1, 3, 2, 8, 4, 9], 2))

ngày 0 giá 1: cash = 0, hold = -1  (mua ngay: hold = -1)
ngày 1 giá 3: bán?   0 vs cash   0 ->   0 (giữ) | mua?  -3 vs hold  -1 ->  -1 (giữ)
ngày 2 giá 2: bán?  -1 vs cash   0 ->   0 (giữ) | mua?  -2 vs hold  -1 ->  -1 (giữ)
ngày 3 giá 8: bán?   5 vs cash   0 ->   5 (bán) | mua?  -8 vs hold  -1 ->  -1 (giữ)
ngày 4 giá 4: bán?   1 vs cash   5 ->   5 (giữ) | mua?   1 vs hold  -1 ->   1 (mua)
ngày 5 giá 9: bán?   8 vs cash   5 ->   8 (bán) | mua?  -4 vs hold   1 ->   1 (giữ)
kết quả: 8


Đọc trace: ở ngày 3 (giá `8`) `bán?` ra `5 > cash = 0` nên chọn **bán** (`cash = 5`); ở ngày 4 (giá `4`) `mua?` = `5 - 4 = 1 > hold = -1` nên **mua lại** (`hold = 1`); ngày 5 (giá `9`) bán tiếp: `1 + 9 - 2 = 8`.

### 8.2. Trace cách 2 (đệ quy có nhớ)

Ta in từng lần **thực sự tính** một trạng thái `(i, holding)` (lần gọi sau trả từ cache). Ví dụ nhỏ `[1, 3, 2]`, `fee = 1`:

In [18]:
def trace_memo(prices, fee):
    n = len(prices)
    memo = {}

    def go(i, holding, depth=0):
        pad = "  " * depth
        if i == n:
            return 0
        if (i, holding) in memo:
            print(f"{pad}go({i}, {holding}) -> lấy từ cache = {memo[(i, holding)]}")
            return memo[(i, holding)]
        print(f"{pad}tính go({i}, {holding}) ...")
        best = go(i + 1, holding, depth + 1)
        if holding:
            best = max(best, prices[i] - fee + go(i + 1, False, depth + 1))
        else:
            best = max(best, -prices[i] + go(i + 1, True, depth + 1))
        memo[(i, holding)] = best
        print(f"{pad}go({i}, {holding}) = {best}")
        return best

    return go(0, False)

print("kết quả:", trace_memo([1, 3, 2], 1))

tính go(0, False) ...
  tính go(1, False) ...
    tính go(2, False) ...
    go(2, False) = 0
    tính go(2, True) ...
    go(2, True) = 1
  go(1, False) = 0
  tính go(1, True) ...
    go(2, True) -> lấy từ cache = 1
    go(2, False) -> lấy từ cache = 0
  go(1, True) = 2
go(0, False) = 1
kết quả: 1


Quan sát: `go(2, False)` và `go(2, True)` chỉ được **tính một lần** rồi lấy từ cache khi các nhánh khác cần tới. Nếu không có cache, ta sẽ tính lại hàng loạt lần.

### 8.3. Trace cách 5 (tham lam)

Phần in `verbose=True` ở mục 7.4 cho ví dụ 1; dưới đây trace chi tiết hơn cho `[1,3,7,5,10,3]`, `fee = 3` (ví dụ 2, đáp án `6`):

In [19]:
def trace_greedy(prices, fee):
    buy, profit = prices[0] + fee, 0
    print(f"khởi tạo: buy = {prices[0]} + {fee} = {buy}, profit = 0")
    for p in prices[1:]:
        if p + fee < buy:
            buy = p + fee
            print(f"giá {p:>2}: {p} + {fee} < buy -> mua rẻ hơn, buy = {buy}")
        elif p > buy:
            profit += p - buy
            print(f"giá {p:>2}: {p} > buy ({buy}) -> chốt lãi {p - buy}, profit = {profit}, đặt lại buy = {p}")
            buy = p
        else:
            print(f"giá {p:>2}: không làm gì (buy = {buy})")
    return profit

print("kết quả:", trace_greedy([1, 3, 7, 5, 10, 3], 3))

khởi tạo: buy = 1 + 3 = 4, profit = 0
giá  3: không làm gì (buy = 4)
giá  7: 7 > buy (4) -> chốt lãi 3, profit = 3, đặt lại buy = 7
giá  5: không làm gì (buy = 7)
giá 10: 10 > buy (7) -> chốt lãi 3, profit = 6, đặt lại buy = 10
giá  3: 3 + 3 < buy -> mua rẻ hơn, buy = 6
kết quả: 6


Đọc trace: khởi tạo `buy = 1 + 3 = 4`. Giá `3`: không `3 + 3 < 4`, cũng không `3 > 4`. Giá `7 > 4`: chốt lãi `3`, `buy = 7`. Giá `5`: không có gì. Giá `10 > 7`: chốt thêm `3` (đây chính là "hủy lần bán ở `7` và bán ở `10`": tổng lãi `3 + 3 = 6 = 10 - 1 - 3`), `buy = 10`. Giá `3`: `3 + 3 < 10` → `buy = 6`. Tổng `6`.

## 9. Code nộp LeetCode

Chọn **cách 4 (hai biến `cash`, `hold`)**: dễ chứng minh (là DP máy trạng thái), `O(n)` thời gian, `O(1)` bộ nhớ. Đây là code đúng class/method/signature (trên LeetCode `List` có sẵn; ở đây ta đã import ở trên):

In [20]:
class Solution:
    def maxProfit(self, prices: List[int], fee: int) -> int:
        cash = 0                 # tiền tối đa khi KHÔNG cầm cổ phiếu
        hold = -prices[0]        # tiền tối đa khi ĐANG cầm cổ phiếu (đã mua ngay ngày 0)
        for p in prices[1:]:
            cash, hold = max(cash, hold + p - fee), max(hold, cash - p)
        return cash


for prices, fee, expected in examples:
    got = Solution().maxProfit(prices, fee)
    print(prices, fee, "->", got)
    assert got == expected

[1, 3, 2, 8, 4, 9] 2 -> 8
[1, 3, 7, 5, 10, 3] 3 -> 6
[5] 1 -> 0
[1, 5, 10] 1 -> 8


## 10. Vì sao thuật toán đúng

**Đúng đắn của máy trạng thái.** Định nghĩa: sau ngày `i`,

- `cash_i` = lợi nhuận tối đa trong các cách giao dịch hợp lệ từ ngày `0` đến `i` mà **cuối ngày `i` không cầm** cổ phiếu;
- `hold_i` = lợi nhuận tối đa (tính tiền mặt, đã trừ tiền mua) mà cuối ngày `i` **đang cầm** một cổ phiếu.

Chứng minh bằng quy nạp theo `i`:

- **Cơ sở** (`i = 0`): chỉ có thể không làm gì (`cash_0 = 0`) hoặc mua (`hold_0 = -prices[0]`).
- **Bước quy nạp.** Cuối ngày `i` không cầm: hoặc **hôm qua cũng không cầm** và hôm nay không làm gì (`cash_{i-1}`), hoặc **hôm qua đang cầm** và hôm nay bán, nhận `prices[i]`, trả phí (`hold_{i-1} + prices[i] - fee`). Đó là mọi khả năng (không thể mua rồi bán trong cùng ngày mà có lợi), nên `cash_i` là max của hai giá trị. Tương tự, cuối ngày `i` đang cầm: hoặc hôm qua đã cầm và giữ (`hold_{i-1}`), hoặc hôm qua không cầm và hôm nay mua (`cash_{i-1} - prices[i]`).

Vì các trạng thái ngày `i` chỉ phụ thuộc ngày `i-1`, và mỗi trạng thái lấy **max trên mọi khả năng**, kết quả là tối ưu (nguyên lý tối ưu của DP).

**Đáp án là `cash_{n-1}`.** Cầm cổ phiếu dở dang ở cuối không có lợi (đã trả tiền mua mà không thu về), nên `cash_{n-1} ≥ hold_{n-1}` về mặt giá trị có lợi.

**Phí tính một lần.** Ta trừ `fee` **lúc bán**. Mỗi giao dịch có đúng một lần bán nên phí được trừ đúng một lần. (Trừ lúc mua cũng cho cùng đáp án, vì mọi giao dịch mua có lợi đều được bán.)

**Cách 5 (tham lam).** Đặt `buy = cash - hold` (chi phí hiệu dụng của "cổ phiếu ảo" ta đang cầm); khi `p > buy`, bán ngay ra lãi `p - buy`, đồng thời đặt `buy = p` tương đương với việc **cho phép mua lại với giá `p`** (nếu sau đó giá tăng thêm `x` thì tổng lãi lại cộng `x`: giống chưa từng bán). Khi `p + fee < buy` thì mua rẻ hơn. Quan hệ này trùng với DP hai trạng thái nên đúng; ta cũng đối chiếu bằng vét cạn ngẫu nhiên ở mục 12.

**Cách 1, 2, 3** là cùng quan hệ truy hồi, khác nhau ở cách tính (đệ quy, có nhớ, bảng).

## 11. Phân tích Time / Space Complexity

Gọi `n` là số ngày.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Đệ quy vét cạn | O(2^n) | O(n) | chỉ làm oracle, `n ≤ ~20` |
| 2. Đệ quy có nhớ | O(n) | O(n) | đệ quy sâu `n`, cần nâng giới hạn |
| 3. Bảng `cash[]`, `hold[]` | O(n) | O(n) | dễ debug, in cả bảng |
| 4. Hai biến | O(n) | O(1) | **bản nộp**, ngắn và an toàn |
| 5. Tham lam `buy` | O(n) | O(1) | gọn nhất nhưng khó chứng minh |

**Chi tiết.** Có `2n` trạng thái `(i, holding)` và mỗi trạng thái có tối đa 2 chuyển (giữ hoặc đổi), nên `O(n)`. Với `n = 5 × 10^4`, các cách `O(n)` xong trong vài chục ms.

**Vì sao không ra `O(n²)`?** Một cách nghĩ khác là chọn cặp (mua, bán) rồi ghép nhiều giao dịch, cần `O(n²)`. Máy trạng thái tránh được việc nhớ "mua ngày nào" bằng cách chỉ nhớ **tiền tối ưu** của từng trạng thái.

## 12. Bộ test

Ta kiểm tra theo nhiều lớp:

1. Ví dụ của đề cho cả 5 cách + bản nộp.
2. Edge case: `n = 1`, `fee = 0`, phí rất lớn (không giao dịch nào có lãi), giá giảm dần, giá tăng dần, giá đều nhau, biên giá trị.
3. **Phản ví dụ** của các cách làm sai (xem thêm mục 13).
4. Đối chiếu với **oracle độc lập** (chọn các cặp mua – bán không chồng nhau bằng DP `O(n²)` theo cặp, khác hẳn máy trạng thái) trên input ngẫu nhiên có seed.
5. Test lớn `n = 5 × 10^4`.

In [21]:
def oracle_pairs(prices, fee):
    """Oracle độc lập: best[j] = lợi nhuận tối đa dùng các ngày 0..j-1 (best[0] = 0).
    Một giao dịch mua ở i, bán ở j-1 (i <= j-1) tạo lãi prices[j-1] - prices[i] - fee.
    Không dùng máy trạng thái cash/hold."""
    n = len(prices)
    best = [0] * (n + 1)
    for j in range(1, n + 1):
        best[j] = best[j - 1]                              # không bán vào ngày j-1
        for i in range(j):                                 # mua ở i, bán ở j-1
            best[j] = max(best[j], best[i] + prices[j - 1] - prices[i] - fee)
    return best[n]


for prices, fee, expected in examples:
    assert oracle_pairs(prices, fee) == expected
print("oracle khớp ví dụ đề")

solvers = {
    "brute": profit_brute,
    "memo": profit_memo,
    "table": profit_table,
    "two_vars": profit_two_vars,
    "greedy": profit_greedy,
    "Solution2": lambda p, f: Solution2().maxProfit(p, f),
    "Solution3": lambda p, f: Solution3().maxProfit(p, f),
    "Solution4": lambda p, f: Solution4().maxProfit(p, f),
    "Solution5": lambda p, f: Solution5().maxProfit(p, f),
    "Solution": lambda p, f: Solution().maxProfit(p, f),
}
for prices, fee, expected in examples:
    for name, fn in solvers.items():
        assert fn(list(prices), fee) == expected, (name, prices, fee)
print("tất cả cách khớp ví dụ đề:", list(solvers))

oracle khớp ví dụ đề
tất cả cách khớp ví dụ đề: ['brute', 'memo', 'table', 'two_vars', 'greedy', 'Solution2', 'Solution3', 'Solution4', 'Solution5', 'Solution']


In [22]:
# --- Edge case ---
edge_cases = [
    ([7], 0),                                  # n = 1
    ([7], 5),
    ([1, 2], 0),                               # fee = 0, tăng
    ([2, 1], 0),                               # fee = 0, giảm
    ([1, 2], 1),                               # lãi đúng bằng phí -> 0
    ([1, 3], 1),                               # lãi 1
    ([1, 2, 3, 4, 5], 0),                      # tăng dần fee 0
    ([1, 2, 3, 4, 5], 2),                      # tăng dần: chỉ một giao dịch
    ([5, 4, 3, 2, 1], 1),                      # giảm dần -> 0
    ([3, 3, 3, 3], 0),                         # đều nhau
    ([1, 100, 1, 100, 1, 100], 1),             # lên xuống mạnh
    ([1, 100, 1, 100, 1, 100], 98),            # phí vừa đủ hết lãi
    ([1, 49999], 49998),                       # biên giá
    ([49999, 1, 49999], 0),                    # biên giá, fee 0
    ([1, 5, 10], 1),
    ([10, 1, 5, 1, 10], 3),
]
for prices, fee in edge_cases:
    want = oracle_pairs(prices, fee)
    for name, fn in solvers.items():
        if name in ("brute",) and len(prices) > 16:
            continue
        got = fn(list(prices), fee)
        assert got == want, (name, prices, fee, got, want)
    print(prices, "fee =", fee, "->", want)

[7] fee = 0 -> 0
[7] fee = 5 -> 0
[1, 2] fee = 0 -> 1
[2, 1] fee = 0 -> 0
[1, 2] fee = 1 -> 0
[1, 3] fee = 1 -> 1
[1, 2, 3, 4, 5] fee = 0 -> 4
[1, 2, 3, 4, 5] fee = 2 -> 2
[5, 4, 3, 2, 1] fee = 1 -> 0
[3, 3, 3, 3] fee = 0 -> 0
[1, 100, 1, 100, 1, 100] fee = 1 -> 294
[1, 100, 1, 100, 1, 100] fee = 98 -> 3
[1, 49999] fee = 49998 -> 0
[49999, 1, 49999] fee = 0 -> 49998
[1, 5, 10] fee = 1 -> 8
[10, 1, 5, 1, 10] fee = 3 -> 7


**Phản ví dụ.** Hai ý tưởng "trông có vẻ đúng" nhưng sai:

- *Sai A:* bỏ qua phí và cộng mọi mức tăng liên tiếp (lời giải bài "không phí" 122), rồi trừ `fee` đúng một lần.
- *Sai B:* mỗi lần giá tăng thì bán/mua lại, trừ phí cho mỗi cặp tăng liên tiếp (tức quên rằng nên **gộp** các lần tăng thành một giao dịch).

In [23]:
def wrong_sum_increases(prices, fee):
    total = sum(max(0, prices[i + 1] - prices[i]) for i in range(len(prices) - 1))
    return max(0, total - fee)                    # SAI A

def wrong_fee_each_rise(prices, fee):
    total = 0
    for i in range(len(prices) - 1):
        gain = prices[i + 1] - prices[i] - fee    # SAI B: mỗi bước tăng là một giao dịch riêng
        if gain > 0:
            total += gain
    return total

prices, fee = [1, 3, 2, 8, 4, 9], 2
print("đúng :", Solution().maxProfit(prices, fee))                 # 8
print("sai A:", wrong_sum_increases(prices, fee))                   # (2 + 6 + 5) - 2 = 11
print("sai B:", wrong_fee_each_rise(prices, fee))
assert wrong_sum_increases(prices, fee) != 8
assert wrong_fee_each_rise(prices, fee) != 8

prices, fee = [1, 2, 3, 4, 5], 1
print("đúng :", Solution().maxProfit(prices, fee), "| sai B:", wrong_fee_each_rise(prices, fee))   # B ra 0, đúng 3
assert wrong_fee_each_rise(prices, fee) != Solution().maxProfit(prices, fee)

đúng : 8
sai A: 11
sai B: 7
đúng : 3 | sai B: 0


Giải thích: **A** cộng cả đoạn lên `3-1`, `8-2`, `9-4` mà chỉ trừ phí một lần, trong khi ta phải trả phí **cho mỗi giao dịch** thực tế (hai giao dịch trong ví dụ 1). **B** làm ngược lại: trừ phí cho **từng bước tăng nhỏ**, bỏ lỡ việc gộp `1 -> 2 -> 3 -> 4 -> 5` thành **một** giao dịch lãi `4 - 1 = 3`.

**Đối chiếu nhiều input ngẫu nhiên (có seed, đúng ràng buộc).** Ta sinh giá và phí từ nhiều miền khác nhau (miền nhỏ để hay hòa, miền lớn để tiệm cận ràng buộc), `n` từ `1` tới `16` cho brute force.

In [24]:
def random_case(rng, n, pmax, fmax):
    prices = [rng.randint(1, pmax) for _ in range(n)]
    fee = rng.randint(0, fmax)
    return prices, fee


rng = random.Random(714)
count = 0
for _ in range(600):
    n = rng.randint(1, 14)
    pmax, fmax = rng.choice([(5, 3), (10, 6), (50, 20), (49999, 49999)])
    prices, fee = random_case(rng, n, pmax, fmax)
    want = oracle_pairs(prices, fee)
    for name, fn in solvers.items():
        got = fn(list(prices), fee)
        assert got == want, (name, prices, fee, got, want)
    count += 1
print("đối chiếu", count, "input ngẫu nhiên (n <= 14): tất cả cách khớp oracle theo cặp")

# n trung bình, oracle O(n^2) vẫn chạy được; bỏ brute
for _ in range(120):
    n = rng.randint(15, 200)
    pmax, fmax = rng.choice([(5, 3), (20, 10), (1000, 200), (49999, 49999)])
    prices, fee = random_case(rng, n, pmax, fmax)
    want = oracle_pairs(prices, fee)
    for name, fn in solvers.items():
        if name == "brute":
            continue
        assert fn(list(prices), fee) == want, (name, n)
print("120 input n từ 15 đến 200: OK")

# dùng cash cũ hay cash mới khi cập nhật hold đều cho cùng đáp án
def two_vars_sequential(prices, fee):
    cash, hold = 0, -prices[0]
    for p in prices[1:]:
        cash = max(cash, hold + p - fee)
        hold = max(hold, cash - p)                 # dùng cash MỚI
    return cash

for _ in range(300):
    prices, fee = random_case(rng, rng.randint(1, 30), 30, 10)
    assert two_vars_sequential(prices, fee) == Solution().maxProfit(prices, fee)
print("cập nhật tuần tự (cash mới) cho cùng đáp án với cập nhật đồng thời: OK")

đối chiếu 600 input ngẫu nhiên (n <= 14): tất cả cách khớp oracle theo cặp
120 input n từ 15 đến 200: OK
cập nhật tuần tự (cash mới) cho cùng đáp án với cập nhật đồng thời: OK


In [25]:
# Test lớn: n = 5 * 10^4 (chỉ các cách lặp), và so sánh thời gian
rng = random.Random(50000)
prices = [rng.randint(1, 49999) for _ in range(50000)]
fee = 100
ref = profit_two_vars(prices, fee)
for name in ("table", "two_vars", "greedy", "Solution3", "Solution4", "Solution5", "Solution"):
    t0 = time.perf_counter()
    r = solvers[name](list(prices), fee)
    dt = time.perf_counter() - t0
    assert r == ref, name
    print(f"{name:>10}: {dt * 1000:7.2f} ms -> {r}")

# cây giá tăng đều: một giao dịch duy nhất, đáp án biết trước
inc = list(range(1, 50000))
assert Solution().maxProfit(inc, 5) == inc[-1] - inc[0] - 5
assert Solution5().maxProfit(inc, 5) == inc[-1] - inc[0] - 5
print("giá tăng đều 1..49999, fee = 5 -> một giao dịch:", Solution().maxProfit(inc, 5))

     table:   25.56 ms -> 413616902
  two_vars:    9.10 ms -> 413616902
    greedy:   18.12 ms -> 413616902
 Solution3:   20.00 ms -> 413616902
 Solution4:    8.55 ms -> 413616902
 Solution5:    4.14 ms -> 413616902
  Solution:    6.08 ms -> 413616902
giá tăng đều 1..49999, fee = 5 -> một giao dịch: 49993


## 13. Các lỗi người mới hay gặp

### Lỗi 1: trừ phí **hai lần** (cả khi mua lẫn khi bán)

Đề: *"phí chỉ tính một lần cho mỗi giao dịch (mua + bán)"*. Nếu trừ cả lúc mua lẫn lúc bán thì mỗi giao dịch mất `2 × fee`.

In [26]:
def wrong_double_fee(prices, fee):
    cash, hold = 0, -prices[0] - fee            # trừ phí khi mua
    for p in prices[1:]:
        cash, hold = max(cash, hold + p - fee), max(hold, cash - p - fee)   # và lại trừ khi bán/mua
    return cash

prices, fee = [1, 3, 2, 8, 4, 9], 2
print("sai  :", wrong_double_fee(prices, fee))
print("đúng :", Solution().maxProfit(prices, fee))

sai  : 4
đúng : 8


**Vì sao sai.** Mỗi giao dịch bị trừ `2 × fee = 4` thay vì `2`, nên ví dụ 1 ra `4` thay vì `8`. **Cách sửa:** chọn **một** thời điểm tính phí (lúc bán, hoặc lúc mua) và bám theo.

### Lỗi 2: khởi tạo `hold = 0` (cổ phiếu "miễn phí")

`hold` là **tiền tối đa khi đang cầm** cổ phiếu, nên sau khi mua ngày 0 phải là `-prices[0]`, không phải `0`.

In [27]:
def wrong_hold_zero(prices, fee):
    cash, hold = 0, 0                           # SAI: coi như được tặng cổ phiếu
    for p in prices[1:]:
        cash, hold = max(cash, hold + p - fee), max(hold, cash - p)
    return cash

prices, fee = [5, 9], 1
print("sai  :", wrong_hold_zero(prices, fee))   # 8: lãi "bán 9 - phí" mà chưa tốn tiền mua
print("đúng :", Solution().maxProfit(prices, fee))   # 3

sai  : 8
đúng : 3


**Vì sao sai.** Với `hold = 0` ta bán được cổ phiếu "tự nhiên có" ở giá `9` và thu `9 - 1 = 8`, chưa hề trừ tiền mua `5`. **Cách sửa:** `hold = -prices[0]` (hoặc `-inf` trước ngày đầu rồi để vòng lặp tự mua).

### Lỗi 3: trả `hold` (hoặc `max(cash, hold)`) thay vì `cash`

Đáp án là tiền khi **không cầm** cổ phiếu cuối cùng.

In [28]:
def wrong_return_hold(prices, fee):
    cash, hold = 0, -prices[0]
    for p in prices[1:]:
        cash, hold = max(cash, hold + p - fee), max(hold, cash - p)
    return hold

prices, fee = [1, 3, 2, 8, 4, 9], 2
print("sai  :", wrong_return_hold(prices, fee), "(hold là tiền khi còn cầm cổ phiếu chưa bán)")
print("đúng :", Solution().maxProfit(prices, fee))

sai  : 1 (hold là tiền khi còn cầm cổ phiếu chưa bán)
đúng : 8


**Vì sao sai.** `hold = 1` nghĩa là "tiền ròng nếu đang cầm cổ phiếu", một giá trị trung gian chứ chưa chốt lãi. **Cách sửa:** `return cash`.

### Lỗi 3b: quên khoảng "bán rồi mua lại ngay cùng ngày" hoặc tính nhầm thứ tự cập nhật

Nếu cập nhật `hold` bằng `cash` **mới** thì kết quả vẫn đúng (xem mục 12), nhưng nếu cập nhật sai thứ tự theo kiểu `hold` trước rồi `cash` dùng `hold` **mới**, ta có thể mua và bán trong cùng một ngày mà không mất phí đúng cách.

In [29]:
def wrong_order(prices, fee):
    cash, hold = 0, -prices[0]
    for p in prices[1:]:
        hold = max(hold, cash - p)              # hold cập nhật trước
        cash = max(cash, hold + p - fee)        # rồi cash dùng hold MỚI: có thể mua và bán cùng ngày
    return cash

# với fee = 0 mua-bán cùng ngày không sinh lãi nên vô hại; thử với fee âm hypothetical để thấy cấu trúc:
prices, fee = [4, 4, 4], 0
print(wrong_order(prices, fee), Solution().maxProfit(prices, fee))       # 0 0 (vô hại với fee >= 0)
for _ in range(500):
    pr, f = random_case(random.Random(_), random.randint(1, 15), 20, 10)
    assert wrong_order(pr, f) == Solution().maxProfit(pr, f)
print("với fee >= 0, thứ tự cập nhật không đổi đáp án (mua-bán cùng ngày không bao giờ có lợi)")

0 0
với fee >= 0, thứ tự cập nhật không đổi đáp án (mua-bán cùng ngày không bao giờ có lợi)


**Kết luận.** Thứ tự cập nhật **không làm sai kết quả** với `fee ≥ 0` (vì mua rồi bán cùng ngày chỉ mất phí), nhưng nó sẽ gây sai ở các bài biến thể có **ngày nghỉ (cooldown)** hay **giới hạn số giao dịch**. Hãy tập thói quen cập nhật đồng thời bằng phép gán tuple.

### Lỗi 4: dùng lời giải "không phí" rồi trừ phí cuối cùng

Cộng mọi mức tăng và trừ phí một lần (hoặc theo mỗi bước) đều sai — ta đã chứng minh ở mục 12 (`wrong_sum_increases`, `wrong_fee_each_rise`).

### Lỗi 5: tham lam quên "đặt lại `buy = p`" sau khi chốt

Trong cách 5, sau khi `profit += p - buy` mà quên `buy = p` thì lần tăng giá sau sẽ cộng lại phần chênh cũ.

In [30]:
def wrong_greedy_no_reset(prices, fee):
    buy, profit = prices[0] + fee, 0
    for p in prices[1:]:
        if p + fee < buy:
            buy = p + fee
        elif p > buy:
            profit += p - buy                   # SAI: quên buy = p
    return profit

prices, fee = [1, 5, 10], 1
print("sai  :", wrong_greedy_no_reset(prices, fee))      # 3 + 8 = 11
print("đúng :", Solution().maxProfit(prices, fee))       # 8
assert wrong_greedy_no_reset(prices, fee) != Solution().maxProfit(prices, fee)

sai  : 11
đúng : 8


**Vì sao sai.** Với `buy = 2` cố định: giá `5` thêm `3`, giá `10` lại thêm `8` (tính lại cả phần `5 - 2` đã chốt) → `11`, lớn hơn đáp án thật `8`. **Cách sửa:** sau khi chốt, đặt `buy = p` để lần sau chỉ cộng phần **tăng thêm** (`10 - 5 = 5`), cho tổng `3 + 5 = 8`.

### Lỗi 6: đệ quy có nhớ với `n` lớn → `RecursionError`

Với `n = 5 × 10^4` mà quên nâng giới hạn đệ quy, bản đệ quy sẽ sập.

In [31]:
def memo_no_limit(prices, fee):                 # KHÔNG nâng giới hạn
    n = len(prices)

    @lru_cache(maxsize=None)
    def go(i, holding):
        if i == n:
            return 0
        best = go(i + 1, holding)
        if holding:
            best = max(best, prices[i] - fee + go(i + 1, False))
        else:
            best = max(best, -prices[i] + go(i + 1, True))
        return best

    return go(0, False)

rng = random.Random(6)
prices = [rng.randint(1, 100) for _ in range(3000)]
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())
try:
    memo_no_limit(prices, 2)
    print("không lỗi (môi trường cho phép sâu hơn)")
except RecursionError:
    print("RecursionError với n = 3000")
print("bản lặp (hai biến):", Solution().maxProfit(prices, 2))

giới hạn đệ quy mặc định: 1000
RecursionError với n = 3000
bản lặp (hai biến): 49386


**Cách sửa.** Dùng DP lặp (cách 3, 4) hoặc `sys.setrecursionlimit(...)`. Với đề này, bản hai biến là lựa chọn an toàn nhất.

## 14. Mẹo & nhận dạng pattern

- **"Mua bán cổ phiếu" + "nhiều giao dịch" + "một chi phí cố định / cooldown / giới hạn `k` lần"** → **máy trạng thái** (`cash`/`hold`, thêm trạng thái `cooldown` nếu có nghỉ, thêm chiều `k` nếu có giới hạn giao dịch).
- **Viết ra trạng thái và chuyển trạng thái trước khi code**: "đang cầm / không cầm", mỗi ngày chọn giữ hoặc đổi.
- **Chọn nơi trừ phí một lần** (lúc bán hoặc lúc mua) và bám theo.
- **Khởi tạo bằng `-prices[0]` hoặc `-inf`** cho trạng thái "đang cầm"; **không** dùng `0`.
- **Gán tuple đồng thời** cho các biến trạng thái phụ thuộc lẫn nhau.
- **Kiểm chứng tham lam bằng vét cạn**: tham lam của bài này ngắn nhưng khó thấy đúng; hãy luôn có oracle `O(2^n)` hoặc `O(n²)`.
- Họ hàng: **121** (một giao dịch), **122** (không giới hạn, không phí), **123/188** (tối đa `k` giao dịch), **309** (có cooldown), **714** (có phí).

## 15. Biến thể

### 15.1. Một giao dịch duy nhất (LeetCode 121)

Giữ giá thấp nhất đã gặp, lãi tối đa là `giá - giá thấp nhất` lớn nhất.

In [32]:
def max_profit_one(prices):
    low, best = float("inf"), 0
    for p in prices:
        low = min(low, p)
        best = max(best, p - low)
    return best

print(max_profit_one([7, 1, 5, 3, 6, 4]))          # 5
assert max_profit_one([7, 1, 5, 3, 6, 4]) == 5
assert max_profit_one([7, 6, 4, 3, 1]) == 0

5


### 15.2. Nhiều giao dịch, không phí (LeetCode 122)

Chính là bài này với `fee = 0`; đáp án bằng tổng mọi mức tăng liên tiếp.

In [33]:
def max_profit_unlimited(prices):
    return sum(max(0, prices[i + 1] - prices[i]) for i in range(len(prices) - 1))

for _ in range(200):
    pr, _f = random_case(random.Random(_), random.randint(1, 20), 30, 0)
    assert max_profit_unlimited(pr) == Solution().maxProfit(pr, 0)
print(max_profit_unlimited([7, 1, 5, 3, 6, 4]))      # 7

7


### 15.3. Có **ngày nghỉ (cooldown)** một ngày sau khi bán (LeetCode 309) — kèm phí

Cần thêm trạng thái "vừa bán hôm nay, hôm sau không được mua". Gọi `hold`, `sold` (vừa bán hôm nay), `rest` (không cầm, được mua):

In [34]:
def max_profit_cooldown(prices, fee=0):
    hold, sold, rest = -prices[0], float("-inf"), 0
    for p in prices[1:]:
        hold, sold, rest = max(hold, rest - p), hold + p - fee, max(rest, sold)
    return max(sold, rest)

print(max_profit_cooldown([1, 2, 3, 0, 2]))          # 3
assert max_profit_cooldown([1, 2, 3, 0, 2]) == 3
assert max_profit_cooldown([1]) == 0
assert max_profit_cooldown([1, 3, 2, 8, 4, 9], 2) <= Solution().maxProfit([1, 3, 2, 8, 4, 9], 2)   # có cooldown thì không hơn

3


### 15.4. Tối đa `k` giao dịch có phí

Thêm chiều `k`: `cash[t]`, `hold[t]` = tối đa tiền khi đã dùng `t` giao dịch. Với `k` đủ lớn, đáp án trùng bài gốc.

In [35]:
def max_profit_at_most_k(prices, k, fee=0):
    NEG = float("-inf")
    cash = [0] * (k + 1)                              # cash[t]: dùng tối đa t giao dịch, không cầm
    hold = [NEG] * (k + 1)                            # hold[t]: đang cầm, đã bắt đầu t giao dịch
    for p in prices:
        for t in range(k, 0, -1):
            cash[t] = max(cash[t], hold[t] + p - fee)         # bán: kết thúc giao dịch thứ t
            hold[t] = max(hold[t], cash[t - 1] - p)           # mua: bắt đầu giao dịch thứ t
    return max(cash)

print(max_profit_at_most_k([3, 2, 6, 5, 0, 3], 2))   # 7 (LeetCode 188 ví dụ 2)
assert max_profit_at_most_k([3, 2, 6, 5, 0, 3], 2) == 7
assert max_profit_at_most_k([2, 4, 1], 2) == 2
rng = random.Random(154)
for _ in range(200):
    pr, f = random_case(rng, rng.randint(1, 12), 20, 5)
    assert max_profit_at_most_k(pr, len(pr), f) == Solution().maxProfit(pr, f)
print("k >= n/2 trùng đáp án bài gốc: OK")

7
k >= n/2 trùng đáp án bài gốc: OK


### 15.5. Liệt kê **các giao dịch** đạt lãi tối đa (cho `n` nhỏ)

Dùng đệ quy có nhớ nhưng trả về cả danh sách giao dịch `(ngày mua, ngày bán)`.

In [36]:
def best_transactions(prices, fee):
    n = len(prices)

    @lru_cache(maxsize=None)
    def go(i, buy_day):                       # buy_day = -1 nếu không cầm
        if i == n:
            return 0, ()
        best = go(i + 1, buy_day)             # không làm gì
        if buy_day >= 0:                      # đang cầm: có thể bán
            sub = go(i + 1, -1)
            cand = (prices[i] - prices[buy_day] - fee + sub[0], ((buy_day, i),) + sub[1])
            if cand[0] > best[0]:
                best = cand
        else:                                 # không cầm: có thể mua
            cand = go(i + 1, i)
            if cand[0] > best[0]:
                best = cand
        return best

    profit, trades = go(0, -1)
    return profit, list(trades)


pr = [1, 3, 2, 8, 4, 9]
profit, trades = best_transactions(pr, 2)
print(profit, trades)                                  # 8 [(0, 3), (4, 5)]
assert profit == 8
assert sum(pr[j] - pr[i] - 2 for i, j in trades) == profit
assert all(trades[k][1] < trades[k + 1][0] or trades[k][1] <= trades[k + 1][0] for k in range(len(trades) - 1))

8 [(0, 3), (4, 5)]


Với `n` lớn người ta dùng bảng DP kèm mảng "đến từ đâu" để truy vết; ở đây ta chỉ cần ý tưởng: **trạng thái nào cũng có thể mang theo đường đi tối ưu của nó**.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đệ quy vét cạn | thử mọi cách mua/bán | O(2^n) | O(n) | làm oracle, `n ≤ ~20` |
| 2. Đệ quy có nhớ | trạng thái `(i, holding)` | O(n) | O(n) | khi muốn code sát lập luận đệ quy; `n` vừa |
| 3. Bảng `cash[]`/`hold[]` | bottom-up hai mảng | O(n) | O(n) | cần truy vết hoặc in bảng |
| 4. Hai biến `cash`, `hold` | rút gọn bảng | O(n) | O(1) | **bản nộp mặc định** |
| 5. Tham lam `buy`/`profit` | giá mua hiệu dụng, "hủy bán" | O(n) | O(1) | khi cần code cực ngắn; nhớ kiểm chứng |

**Lời khuyên.** Học cách 4 (máy trạng thái) trước vì nó dùng lại được cho cả họ bài cổ phiếu (121, 122, 123, 188, 309). Cách 5 đáng biết nhưng khó mở rộng.

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy:

In [37]:
print(Solution().maxProfit([1, 3, 2, 8, 4, 9], 2))
print(Solution().maxProfit([5, 4, 3], 1))
print(Solution().maxProfit([2, 6], 1))

8
0
3


**Bài 2 (tính tay).** Với `prices = [1, 5, 3, 8]`, `fee = 1`, tính tay `(cash, hold)` sau từng ngày theo công thức của mục 3.2 (ngày 0: `cash = 0`, `hold = -1`) rồi gán lợi nhuận tối đa vào `ans2` (đang là `None`).

In [38]:
ans2 = None      # TODO: một số nguyên

**Bài 3 (điền TODO).** Viết hàm `next_state(cash, hold, p, fee)` trả về `(cash_mới, hold_mới)` sau một ngày giá `p`, **dùng giá trị cũ của cả hai** (như hàm `step` ở mục 3.2 nhưng bạn tự viết lại).

In [39]:
def next_state(cash, hold, p, fee):
    # TODO
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_profit(prices, fee)` theo cách **tham lam giá mua hiệu dụng** (cách 5).

In [40]:
def my_profit(prices, fee):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) `profit_cooldown_fee(prices, fee)`: như bài gốc nhưng **sau khi bán phải nghỉ đúng một ngày** (hôm sau không được mua). (b) `profit_with_max_trades(prices, fee, k)`: **tối đa `k` giao dịch**, mỗi giao dịch trả phí `fee`.

In [41]:
def profit_cooldown_fee(prices, fee):
    # TODO
    pass

def profit_with_max_trades(prices, fee, k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: giá giảm dần thì không nên mua gì.
- Bài 2: ngày 0 `(0, -1)`; ngày 1 giá `5`...
- Bài 3: hai công thức max ở mục 3.2.
- Bài 4: `buy = prices[0] + fee`, hai nhánh `p + fee < buy` và `p > buy`.
- Bài 5: (a) cần thêm một trạng thái; (b) cần thêm chiều `t` (số giao dịch đã dùng).

**Mức 2 (vừa):**
- Bài 2: ngày 1: `cash = max(0, -1 + 5 - 1) = 3`, `hold = max(-1, 0 - 5) = -1`. Ngày 2 (giá 3): `cash = max(3, -1 + 3 - 1) = 3`, `hold = max(-1, 3 - 3) = 0`. Ngày 3 (giá 8): `cash = max(3, 0 + 8 - 1) = 7`.
- Bài 3: `return max(cash, hold + p - fee), max(hold, cash - p)`.
- Bài 4: `profit += p - buy; buy = p` khi chốt.
- Bài 5: (a) `hold`, `sold` (vừa bán), `rest` (không cầm, mua được) như mục 15.3, thêm trừ phí khi bán; (b) như 15.4 nhưng trừ `fee` khi bán.

**Mức 3 (gần đáp án):**
- Bài 3: `new_cash = max(cash, hold + p - fee)` và `new_hold = max(hold, cash - p)`, tính cả hai trước khi gán.
- Bài 4: xem cách 5 ở mục 7.4.
- Bài 5: (a) `hold, sold, rest = max(hold, rest - p), hold + p - fee, max(rest, sold)`, đáp án `max(sold, rest)`; (b) xem `max_profit_at_most_k` ở mục 15.4.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [42]:
# Bài 1
assert Solution().maxProfit([1, 3, 2, 8, 4, 9], 2) == 8
assert Solution().maxProfit([5, 4, 3], 1) == 0
assert Solution().maxProfit([2, 6], 1) == 3

# Bài 2: ngày 0 (0,-1); ngày 1 (3,-1); ngày 2 (3,0); ngày 3 (7,0) => 7
ans2 = 7
assert ans2 == Solution().maxProfit([1, 5, 3, 8], 1) == oracle_pairs([1, 5, 3, 8], 1)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [43]:
# Bài 3
def next_state(cash, hold, p, fee):
    return max(cash, hold + p - fee), max(hold, cash - p)

assert next_state(0, float("-inf"), 1, 2) == (0, -1)
assert next_state(0, -1, 3, 2) == (0, -1)
assert next_state(0, -1, 8, 2) == (5, -1)
assert next_state(5, -1, 4, 2) == (5, 1)
cash, hold = 0, float("-inf")
for p in [1, 3, 2, 8, 4, 9]:
    cash, hold = next_state(cash, hold, p, 2)
assert cash == 8
print("Bài 3 OK")

Bài 3 OK


In [44]:
# Bài 4
def my_profit(prices, fee):
    buy, profit = prices[0] + fee, 0
    for p in prices[1:]:
        if p + fee < buy:
            buy = p + fee
        elif p > buy:
            profit += p - buy
            buy = p
    return profit

rng = random.Random(4)
for _ in range(300):
    pr, f = random_case(rng, rng.randint(1, 14), rng.choice([5, 20, 49999]), rng.choice([0, 3, 49999]))
    assert my_profit(pr, f) == oracle_pairs(pr, f)
print("Bài 4 OK")

Bài 4 OK


In [45]:
# Bài 5 (a)
def profit_cooldown_fee(prices, fee):
    hold, sold, rest = -prices[0], float("-inf"), 0
    for p in prices[1:]:
        hold, sold, rest = max(hold, rest - p), hold + p - fee, max(rest, sold)
    return max(sold, rest)

# oracle vét cạn cho cooldown
def oracle_cooldown(prices, fee):
    n = len(prices)

    def go(i, holding, blocked):
        if i >= n:
            return 0
        best = go(i + 1, holding, False)                       # không làm gì (hết cooldown nếu có)
        if holding:
            best = max(best, prices[i] - fee + go(i + 1, False, True))   # bán rồi hôm sau nghỉ
        elif not blocked:
            best = max(best, -prices[i] + go(i + 1, True, False))
        return best

    return go(0, False, False)

assert profit_cooldown_fee([1, 2, 3, 0, 2], 0) == 3
assert profit_cooldown_fee([1], 3) == 0
rng = random.Random(51)
for _ in range(300):
    pr, f = random_case(rng, rng.randint(1, 12), 15, 5)
    assert profit_cooldown_fee(pr, f) == oracle_cooldown(pr, f), (pr, f)

# Bài 5 (b)
def profit_with_max_trades(prices, fee, k):
    NEG = float("-inf")
    cash = [0] * (k + 1)
    hold = [NEG] * (k + 1)
    for p in prices:
        for t in range(k, 0, -1):
            cash[t] = max(cash[t], hold[t] + p - fee)
            hold[t] = max(hold[t], cash[t - 1] - p)
    return max(cash)

def oracle_k(prices, fee, k):
    n = len(prices)

    def go(i, holding, left):
        if i == n:
            return 0
        best = go(i + 1, holding, left)
        if holding:
            best = max(best, prices[i] - fee + go(i + 1, False, left))
        elif left > 0:
            best = max(best, -prices[i] + go(i + 1, True, left - 1))
        return best

    return go(0, False, k)

assert profit_with_max_trades([3, 2, 6, 5, 0, 3], 0, 2) == 7
assert profit_with_max_trades([1, 3, 2, 8, 4, 9], 2, 1) == 6          # 1 giao dịch: 9-1-2 = 6
assert profit_with_max_trades([1, 3, 2, 8, 4, 9], 2, 2) == 8
rng = random.Random(52)
for _ in range(300):
    pr, f = random_case(rng, rng.randint(1, 12), 15, 4)
    k = rng.randint(0, 4)
    assert profit_with_max_trades(pr, f, k) == oracle_k(pr, f, k), (pr, f, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (máy trạng thái hai biến):**

```python
class Solution:
    def maxProfit(self, prices: List[int], fee: int) -> int:
        cash = 0                    # không cầm cổ phiếu
        hold = -prices[0]           # đang cầm cổ phiếu (đã mua ngày 0)
        for p in prices[1:]:
            cash, hold = max(cash, hold + p - fee), max(hold, cash - p)
        return cash
```

**Template (tham lam `buy`/`profit`):**

```python
buy, profit = prices[0] + fee, 0
for p in prices[1:]:
    if p + fee < buy:
        buy = p + fee               # mua rẻ hơn
    elif p > buy:
        profit += p - buy           # chốt lãi
        buy = p                     # chỉ tính phần tăng thêm từ giá này
return profit
```

**Câu thần chú:** *hai trạng thái cash/hold; bán thì trừ phí một lần; đáp án là cash; hold khởi tạo là `-prices[0]`.*

**Checklist trước khi submit:**

- [ ] Có đúng hai trạng thái (`cash`, `hold`) và đáp án là `cash`?
- [ ] Phí chỉ trừ **một lần** mỗi giao dịch?
- [ ] `hold` khởi tạo `-prices[0]` (hoặc `-inf`), không phải `0`?
- [ ] Cập nhật `cash` và `hold` bằng giá trị **cũ** (gán tuple)?
- [ ] Không dùng đệ quy sâu `5 × 10^4` (hoặc đã nâng giới hạn)?
- [ ] Nếu dùng tham lam: có `buy = p` sau khi chốt?
- [ ] Đã thử: `n = 1`, `fee = 0`, phí rất lớn, giá giảm dần, giá tăng dần, ví dụ của đề?